# Phase 5 — does the geometry laterality route hold at corpus scale?

Laterality is unresolved for ~49% of studies, so half the coronal series are never mirrored to
canonical and medial/lateral is ambiguous for them. That is the standing suspect for MCL scoring
0.503 (random) in Phase 4 run 1.

Phase 0 rejected an `ImagePositionPatient`-sign fallback at 40% disagreement with the real tag. That
measurement was of the **corner**: `ImagePositionPatient` is the first voxel, up to half a field of
view from the anatomy, so on a knee scanned near the midline it sits on the far side of x=0 from the
knee. Walking to the image **centre** first removes that offset. On the 11 tagged series in the
local sample the corner form agrees 9/11 and the centre form 11/11, with both corner failures at
|p_x| < 20 mm — exactly the predicted case.

n=11 decides nothing. This kernel runs the centre form over all 4,407 studies and scores it against
the ~51% that carry a real `Laterality`/`ImageLaterality` tag. **This project has already been
burned once by trusting a DICOM heuristic from a small sample, which is the whole reason the corner
form got as far as the plan document.**

Header-only, CPU-only. Nothing is re-prepped on the strength of this until the number is in.

In [ ]:
import glob, os, shutil, sys, time

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

from knee.dicom import (
    _read_slice_header,
    image_centre_x,
    read_laterality_header,
    resolve_study_laterality,
    side_from_geometry,
)

# Fail loudly if the src dataset predates the geometry route rather than silently
# measuring nothing -- this kernel exists only to test that function.
assert callable(side_from_geometry), 'src dataset is older than the geometry route'
print('src ok:', SRC)

In [ ]:
from pathlib import Path
import pandas as pd

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407, len(all_uids)
TRAIN_SERIES_DIR = Path(f'{COMP_DIR}/train_series')
print(f'{len(all_uids)} studies to walk')

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def read_study(study_uid):
    """One representative slice per series: the geometry fields side_from_geometry
    needs and the tags resolve_study_laterality needs, from the same file."""
    study_dir = TRAIN_SERIES_DIR / study_uid
    geom_headers, tag_headers = [], []
    if not study_dir.is_dir():
        return {'StudyInstanceUID': study_uid, 'n_series': 0,
                'tag_side': None, 'tag_route': 'no_study_dir',
                'geom_side': None, 'geom_route': 'no_study_dir', 'centre_x': None}
    for series_dir in sorted(p for p in study_dir.iterdir() if p.is_dir()):
        dcm_files = sorted(series_dir.glob('*.dcm'))
        if not dcm_files:
            continue
        first = dcm_files[0]
        try:
            geom_headers.append(_read_slice_header(first))
            tag_headers.append(read_laterality_header(first))
        except Exception:
            # a slice that will not parse is one series lost, not one study lost
            continue
    tag_side, tag_route = resolve_study_laterality(tag_headers)
    geom_side, geom_route = side_from_geometry(geom_headers)
    centres = [c for c in (image_centre_x(h) for h in geom_headers) if c is not None]
    return {
        'StudyInstanceUID': study_uid,
        'n_series': len(geom_headers),
        'tag_side': tag_side, 'tag_route': tag_route,
        'geom_side': geom_side, 'geom_route': geom_route,
        'centre_x': float(pd.Series(centres).median()) if centres else None,
    }

# The mount is latency-bound rather than CPU-bound on header-only reads, so threads
# help here where they did not help the prep loop (NOTES 2026-08-10).
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as pool:
    rows = list(pool.map(read_study, all_uids))
print(f'walked {len(rows)} studies in {(time.time()-t0)/60:.1f} min')

lat = pd.DataFrame(rows)
lat.to_csv('/kaggle/working/laterality_geometry_check.csv', index=False)
print(lat['geom_route'].value_counts().to_string())

## The number this kernel exists for

Agreement is scored **only where a real tag exists**, which is the only ground truth available.
Coverage is reported separately: a route that agrees perfectly but resolves nothing is worthless,
and one that resolves everything by guessing is worse than useless.

In [ ]:
both = lat[lat['tag_side'].isin(['L', 'R']) & lat['geom_side'].isin(['L', 'R'])]
agree = int((both['tag_side'] == both['geom_side']).sum())
n = len(both)
print(f'AGREEMENT  {agree}/{n} = {agree/n:.3%}' if n else 'no overlap to score')

tagged = lat['tag_side'].isin(['L', 'R'])
geom = lat['geom_side'].isin(['L', 'R'])
print(f"\ntag route resolves       {tagged.sum():5d} / {len(lat)} = {tagged.mean():.1%}")
print(f"geometry route resolves  {geom.sum():5d} / {len(lat)} = {geom.mean():.1%}")
print(f"either resolves          {(tagged|geom).sum():5d} / {len(lat)} = {(tagged|geom).mean():.1%}")
print(f"NEWLY resolved by geometry {(geom & ~tagged).sum():5d} "
      f"({(geom & ~tagged).mean():.1%} of corpus)")
print(f"still unresolved           {(~(tagged|geom)).sum():5d}")

# Disagreements are the interesting rows: if they cluster near the midline the band
# is too narrow, and if they do not, the rule has a real failure mode worth seeing.
bad = both[both['tag_side'] != both['geom_side']]
if len(bad):
    print(f"\n{len(bad)} disagreements, |centre_x| distribution:")
    print(bad['centre_x'].abs().describe().to_string())
else:
    print('\nno disagreements')

## Gate

Pre-registered before the run, so it cannot be rationalised afterwards. The route is only worth
re-prepping the corpus for if it is both accurate and additive:

- **>= 99% agreement and >= 30% of the corpus newly resolved** — adopt: wire into
  `resolve_laterality` as the fallback after the tag routes, re-prep, retrain.
- **97-99%** — adopt but widen the midline band first, and re-check that agreement recovers.
- **< 97%** — do not adopt. The corner form was rejected at this same table; a route that
  mislabels >3% of knees poisons medial/lateral for those studies rather than fixing it.

In [ ]:
if n:
    acc = agree / n
    newly = (geom & ~tagged).mean()
    print(f'agreement {acc:.3%}, newly resolved {newly:.1%}')
    if acc >= 0.99 and newly >= 0.30:
        print('VERDICT: adopt')
    elif acc >= 0.97:
        print('VERDICT: adopt only after widening the midline band')
    else:
        print('VERDICT: do not adopt')